In [4]:
import subprocess
import pandas as pd
import numpy as np
import xgboost as xgb
import os

def predict_wheeze_wav(input_wav_path, output_csv_path, smile_exe_path, config_path, model_path):
    temp_csv = "temp_features.csv"
    
    # Step 1: Extract features
    cmd = [smile_exe_path, "-C", config_path, "-I", input_wav_path, "-O", temp_csv]
    subprocess.run(cmd, check=True)
    print("Features extracted!")
    
    # Step 2: Semicolon-aware parsing
    df_raw = pd.read_csv(temp_csv, header=None)
    
    if df_raw.shape[1] == 1:
        # Split semicolon-separated values first
        df_semicolon = df_raw[0].str.split(';', expand=True)
        
        # Detect header
        first_row = df_semicolon.iloc[0]
        if any('frameTime' in str(cell) or 'frameIndex' in str(cell) for cell in first_row):
            df_data = df_semicolon.iloc[1:].copy()
            print("Header skipped")
        else:
            df_data = df_semicolon.copy()
        
        # Parse space-separated values within fields
        df_split_list = []
        for _, row in df_data.iterrows():
            split_row = []
            for cell in row.dropna():
                if pd.isna(cell):
                    split_row.append(np.nan)
                else:
                    parts = str(cell).strip().split()
                    split_row.extend(parts[:3])  # Get frameIndex, frameTime, energy
            df_split_list.append(split_row)
        
        df_split = pd.DataFrame(df_split_list)
        while df_split.shape[1] < 3:
            df_split[f'pad_{df_split.shape[1]}'] = np.nan
        df_split.columns = ['frameIndex', 'frameTime', 'pcm_LOGenergy'] + [f'extra_{i}' for i in range(3, df_split.shape[1])]
    else:
        df_split = df_raw.copy()
        df_split.columns = ['frameIndex', 'frameTime', 'pcm_LOGenergy'] + [f'col_{i}' for i in range(3, df_split.shape[1])]
    
    # Numeric conversion
    df_split['frameTime'] = pd.to_numeric(df_split['frameTime'], errors='coerce')
    df_split['pcm_LOGenergy'] = pd.to_numeric(df_split['pcm_LOGenergy'], errors='coerce')
    df_split = df_split.dropna(subset=['frameTime', 'pcm_LOGenergy']).reset_index(drop=True)
    print(f"Parsed {len(df_split)} frames")
    print(df_split.head(3))
    
    # Step 3-5: Features, predict, output (unchanged)
    energy = df_split['pcm_LOGenergy']
    df_split['log_energy'] = np.log1p(np.abs(energy))
    df_split['energy_norm'] = (energy - energy.mean()) / energy.std()
    df_split['time_norm'] = df_split['frameTime'] / df_split['frameTime'].max()
    
    feature_cols = ['log_energy', 'energy_norm', 'time_norm']
    X = df_split[feature_cols]
    
    model = xgb.XGBClassifier()
    model.load_model(model_path)
    predictions = model.predict(X)
    
    output_df = pd.DataFrame({
        'frameTime': df_split['frameTime'].round(2),
        'Wheeze': predictions.astype(int)
    })
    output_df.to_csv(output_csv_path + '.csv', index=False)
    print(f"Saved: {output_csv_path}.csv | Wheeze+: {(predictions == 1).sum()}")
    
    os.remove(temp_csv)
    return output_df

# Your paths (now works!)
predict_wheeze_wav(
    input_wav_path=r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\steth_20181001_11_02_11.wav",
    output_csv_path=r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\trial\trial_1",
    smile_exe_path=r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe",
    config_path=r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\config\demo\demo1_energy.conf",
    model_path=r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\xgb_sentiment_model.json"
)

Features extracted!
Header skipped
Parsed 1498 frames
  frameIndex  frameTime  pcm_LOGenergy
0          0       0.00      -7.290220
1          1       0.01      -6.951265
2          2       0.02      -6.997001
Saved: E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\trial\trial_1.csv | Wheeze+: 657


,frameTime,Wheeze
0,0.00,0
1,0.01,1
2,0.02,1
3,0.03,1
4,0.04,0
...,...,...
1493,14.93,0
1494,14.94,0
1495,14.95,0
1496,14.96,0


In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

def validate_predictions(pred_csv_path, label_txt_path):
    """
    Standalone validation: compare predictions vs ground truth labels
    """
    # Load predictions
    pred_df = pd.read_csv(pred_csv_path)
    print(f"Loaded predictions: {len(pred_df)} frames")
    
    # Load ground truth labels
    label_df = pd.read_csv(label_txt_path, sep='\s+', header=None, names=['start', 'end', 'label'])
    wheeze_intervals = label_df[label_df['label'] == 'Wheeze'][['start', 'end']].values
    print(f"Found {len(wheeze_intervals)} Wheeze intervals")
    
    # Vectorized ground truth assignment (matches training)
    def is_wheeze_vectorized(frametimes, intervals):
        labels = pd.Series(0, index=frametimes.index)
        for start, end in intervals:
            mask = (frametimes >= start) & (frametimes <= end)
            labels[mask] = 1
        return labels
    
    ground_truth = is_wheeze_vectorized(pred_df['frameTime'], wheeze_intervals)
    pred_df['Wheeze_true'] = ground_truth.values
    
    # Validation metrics
    valid_mask = ground_truth.notna()
    acc = accuracy_score(ground_truth[valid_mask], pred_df['Wheeze'][valid_mask])
    prec = precision_score(ground_truth[valid_mask], pred_df['Wheeze'][valid_mask])
    rec = recall_score(ground_truth[valid_mask], pred_df['Wheeze'][valid_mask])
    cm = confusion_matrix(ground_truth[valid_mask], pred_df['Wheeze'][valid_mask])
    
    # Results
    print("\n" + "="*50)
    print("VALIDATION RESULTS")
    print("="*50)
    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"Wheeze+ detected: {pred_df['Wheeze'].sum():4d} / {len(pred_df):4d}")
    print(f"Wheeze+ true:     {ground_truth.sum():4d}")
    print("\nConfusion Matrix:")
    print(f"           Pred0  Pred1")
    print(f"True0    {cm[0,0]:4d}  {cm[0,1]:4d}")
    print(f"True1    {cm[1,0]:4d}  {cm[1,1]:4d}")
    print("="*50)
    
    # Save validated results
    validated_path = pred_csv_path.replace('.csv', '_validated.csv')
    pred_df.to_csv(validated_path, index=False)
    print(f"✓ Saved: {validated_path}")
    
    return pred_df

# Usage (after running prediction)
validate_predictions(
    pred_csv_path=r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\trial\trial_1.csv",
    label_txt_path=r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\steth_20181001_11_02_11_label_audacity.txt"
)


Loaded predictions: 1498 frames
Found 11 Wheeze intervals

VALIDATION RESULTS
Accuracy:  0.380
Precision: 0.393
Recall:    0.327
Wheeze+ detected:  657 / 1498
Wheeze+ true:      788

Confusion Matrix:
           Pred0  Pred1
True0     311   399
True1     530   258
✓ Saved: E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\trial\trial_1_validated.csv


<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
C:\Users\HP\AppData\Local\Temp\ipykernel_3228\637280145.py:13: SyntaxWarning: invalid escape sequence '\s'
  label_df = pd.read_csv(label_txt_path, sep='\s+', header=None, names=['start', 'end', 'label'])


,frameTime,Wheeze,Wheeze_true
0,0.00,0,1
1,0.01,1,1
2,0.02,1,1
3,0.03,1,1
4,0.04,0,1
...,...,...,...
1493,14.93,0,1
1494,14.94,0,1
1495,14.95,0,1
1496,14.96,0,1
